# makemore — A Complete Walkthrough of 6 Language Models

**makemore** takes a text file of words (one per line) and generates new words
that sound like they came from the same distribution.  Fed 32 000 baby names it
invents new ones: *dontell, sherrith, ryel, loghlyn*.  Fed company names it
generates company-like strings.  Same code, different data.

## The one task every model solves

> Given the characters seen so far in a word, output a **probability
> distribution** over what character comes next.

This is **autoregressive character-level language modelling** — the same
principle behind GPT, just at a much smaller scale.

Once you can predict the next character well, generation is trivial:

```
START → sample char₁ → sample char₂ → … → STOP
```

## The six models in this notebook

| # | Model | How it uses context | Key limitation |
|---|-------|---------------------|----------------|
| 1 | **Bigram** | 1 previous character — a lookup table | No history beyond 1 char |
| 2 | **MLP** | Last N chars embedded & concatenated | Fixed context window |
| 3 | **BoW** | Average of all previous embeddings | Loses position information |
| 4 | **RNN** | Sequential hidden state | Gradient vanishing |
| 5 | **GRU** | Gated RNN (learned forget/update) | Slightly more complex |
| 6 | **Transformer** | All chars attend to all previous, in parallel | Needs positional encoding |

Each model fixes a specific weakness of the previous one.  By the end you will
understand *why* the Transformer is built the way it is.

---
**How to read this notebook:**  execute cells top-to-bottom.  Markdown cells
explain the *why*; code cells show the *how*.  Every model section has:
(a) concept, (b) full class, (c) step-by-step forward-pass walkthrough with
shapes, (d) training.


In [ ]:
import os, math, time
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(3407)
torch.cuda.manual_seed_all(3407)
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")


## Part 1 — The Data Pipeline

### What the file looks like

`names.txt` has one name per line, lower-case:

```
emma
olivia
ava
isabella
…
```

32 000 names total.  The vocabulary is 26 letters plus one special token (index
**0**) that serves as both `<START>` and `<STOP>`.

### The autoregressive (X, Y) setup

For the name **"emma"** the dataset produces one training example:

```
X  =  [0,  e,  m,  m,  a ]   ← START token, then each character
Y  =  [e,  m,  m,  a,  0 ]   ← the *next* character at every position
```

At each position *i* the model sees `X[i]` and must predict `Y[i]`.

Padding positions that fall after the word ends are filled with `-1` and masked
out of the loss — the model is not penalised for them.

### Why this encoding?

* **Token 0** is dual-purpose: at the start it says "nothing has been seen yet";
  at the end it is what the model must predict to say "stop generating".
* By processing all positions in one forward pass we get `T` training signals
  from one word instead of one — much more efficient.


In [ ]:
with open('names.txt', 'r') as f:
    data = f.read()

words = data.splitlines()
words = [w.strip() for w in words if w.strip()]

chars = sorted(set(''.join(words)))
max_word_length = max(len(w) for w in words)

print(f"Total words      : {len(words)}")
print(f"Vocabulary size  : {len(chars)} letters + 1 special = {len(chars)+1} tokens")
print(f"Longest word     : {max(words, key=len)!r}  ({max_word_length} chars)")
print(f"Characters       : {''.join(chars)}")
print()
print("First 10 words:", words[:10])


## CharDataset — turning words into tensors

`CharDataset` wraps the word list in a PyTorch `Dataset`.  It handles:

* **Vocabulary** — a mapping `char → int` (`stoi`) and back (`itos`).  Index 0
  is reserved for the special START/STOP token.  Letters are 1-indexed.
* **Encoding** — `encode("emma")` → `tensor([5, 13, 13, 1])`.
* **Decoding** — `decode([5,13,13,1])` → `"emma"`.
* **`__getitem__`** — returns `(X, Y)` tensors of length `max_word_length + 1`.
  Positions after the word end have `Y = -1` (masked out of the loss).


In [ ]:
class CharDataset(Dataset):

    def __init__(self, words, chars, max_word_length):
        self.words = words
        self.chars = chars
        self.max_word_length = max_word_length
        # stoi: char -> index (1-based; 0 is the special token)
        self.stoi = {ch: i+1 for i, ch in enumerate(chars)}
        self.itos = {i: s for s, i in self.stoi.items()}   # reverse mapping

    def __len__(self):
        return len(self.words)

    def contains(self, word):
        return word in self.words

    def get_vocab_size(self):
        return len(self.chars) + 1   # +1 for the special token at index 0

    def get_output_length(self):
        return self.max_word_length + 1   # word + trailing STOP token

    def encode(self, word):
        return torch.tensor([self.stoi[w] for w in word], dtype=torch.long)

    def decode(self, ix):
        return ''.join(self.itos[i] for i in ix)

    def __getitem__(self, idx):
        word = self.words[idx]
        ix   = self.encode(word)           # integer indices of the letters

        # x starts with 0 (START token), then the letters
        x = torch.zeros(self.max_word_length + 1, dtype=torch.long)
        # y is the letters, then 0 (STOP token)
        y = torch.zeros(self.max_word_length + 1, dtype=torch.long)

        x[1 : 1 + len(ix)] = ix
        y[0 : len(ix)]     = ix
        y[len(ix) + 1 :]   = -1   # mask positions after STOP with -1

        return x, y


In [ ]:
# 90% train, 10% test (up to 1000 test examples)
test_set_size = min(1000, int(len(words) * 0.1))
rp = torch.randperm(len(words)).tolist()
train_words = [words[i] for i in rp[:-test_set_size]]
test_words  = [words[i] for i in rp[-test_set_size:]]

train_dataset = CharDataset(train_words, chars, max_word_length)
test_dataset  = CharDataset(test_words,  chars, max_word_length)

vocab_size  = train_dataset.get_vocab_size()
block_size  = train_dataset.get_output_length()

print(f"Train examples : {len(train_dataset)}")
print(f"Test  examples : {len(test_dataset)}")
print(f"vocab_size     : {vocab_size}")
print(f"block_size     : {block_size}  (max word length + 1)")


In [ ]:
# --- inspect one (X, Y) pair ---
word = 'emma'
idx  = train_dataset.words.index(word) if word in train_dataset.words else 0
X_ex, Y_ex = train_dataset[idx]

print(f"Word      : {train_dataset.words[idx]!r}")
print(f"X (raw)   : {X_ex.tolist()}")
print(f"Y (raw)   : {Y_ex.tolist()}")
print()

# decode X, treating 0 as '<S>' and -1 as '<PAD>'
def pretty(t):
    mapping = {0: '<S>', -1: '<PAD>'}
    out = []
    for v in t.tolist():
        out.append(mapping.get(v) or train_dataset.itos[v])
    return out

print(f"X (decoded): {pretty(X_ex)}")
print(f"Y (decoded): {pretty(Y_ex)}")
print()
print("At each position i, the model sees X[i] and must predict Y[i].")
print("Positions with Y=-1 are masked out of the cross-entropy loss.")


## Shared utilities

Three functions are shared across all models:

* **`generate`** — autoregressive sampling loop (no gradient).
* **`evaluate`** — average cross-entropy loss on a dataset.
* **`print_samples`** — draw N samples, bucket them into
  "already in train / in test / genuinely new".
* **`train_model`** — a reusable training loop (replaces the argparse-driven
  `__main__` block in the original script).


In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, do_sample=True, top_k=None):
    '''
    idx : LongTensor (B, T) — conditioning context
    Returns idx extended by max_new_tokens columns.
    '''
    block_size = model.get_block_size()
    for _ in range(max_new_tokens):
        # crop context to block_size if it has grown too long
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        logits, _ = model(idx_cond)
        # take logits at the LAST position only
        logits = logits[:, -1, :] / temperature          # (B, vocab_size)
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        if do_sample:
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            _, idx_next = torch.topk(probs, k=1, dim=-1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


@torch.inference_mode()
def evaluate(model, dataset, device, batch_size=50, max_batches=None):
    model.eval()
    loader = DataLoader(dataset, shuffle=True, batch_size=batch_size, num_workers=0)
    losses = []
    for i, (X, Y) in enumerate(loader):
        X, Y = X.to(device), Y.to(device)
        _, loss = model(X, Y)
        losses.append(loss.item())
        if max_batches is not None and i + 1 >= max_batches:
            break
    model.train()
    return torch.tensor(losses).mean().item()


def print_samples(model, train_dataset, test_dataset, device, num=20, top_k=None):
    model.eval()
    X_init = torch.zeros(num, 1, dtype=torch.long, device=device)
    steps  = train_dataset.get_output_length() - 1
    X_samp = generate(model, X_init, steps, top_k=top_k, do_sample=True).cpu()

    train_set = set(train_dataset.words)
    test_set  = set(test_dataset.words)

    results = {'in train': [], 'in test': [], 'new': []}
    for row in X_samp:
        row  = row[1:].tolist()   # drop the leading START token
        row  = row[:row.index(0)] if 0 in row else row   # crop at STOP
        word = train_dataset.decode(row)
        if word in train_set:
            results['in train'].append(word)
        elif word in test_set:
            results['in test'].append(word)
        else:
            results['new'].append(word)

    print('-' * 60)
    for label, words_ in results.items():
        print(f"{len(words_):3d} samples {label}: {', '.join(words_[:10])}")
    print('-' * 60)
    model.train()


In [ ]:
class InfiniteDataLoader:
    '''Wraps a DataLoader to yield batches indefinitely.'''
    def __init__(self, dataset, **kwargs):
        sampler = torch.utils.data.RandomSampler(
            dataset, replacement=True, num_samples=int(1e10))
        self.loader = DataLoader(dataset, sampler=sampler, **kwargs)
        self._iter  = iter(self.loader)

    def next(self):
        try:
            return next(self._iter)
        except StopIteration:
            self._iter = iter(self.loader)
            return next(self._iter)


def train_model(model, train_dataset, test_dataset, device,
                max_steps=1000, batch_size=32,
                lr=5e-4, weight_decay=0.01, print_every=200):
    '''
    Generic training loop used by every model in this notebook.
    Returns (train_losses, test_losses) recorded at evaluation checkpoints.
    '''
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay,
        betas=(0.9, 0.99), eps=1e-8)
    loader = InfiniteDataLoader(
        train_dataset, batch_size=batch_size, num_workers=0, pin_memory=False)

    train_losses, test_losses, steps_log = [], [], []
    best_loss, t0 = None, time.time()

    for step in range(1, max_steps + 1):
        X, Y = loader.next()
        X, Y = X.to(device), Y.to(device)

        _, loss = model(X, Y)
        model.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % print_every == 0 or step == max_steps:
            tr = evaluate(model, train_dataset, device, max_batches=10)
            te = evaluate(model, test_dataset,  device, max_batches=10)
            train_losses.append(tr)
            test_losses.append(te)
            steps_log.append(step)
            elapsed = time.time() - t0
            print(f"step {step:5d} | train {tr:.4f} | test {te:.4f} | {elapsed:.1f}s")

    return steps_log, train_losses, test_losses


---
## Model 1 — Bigram

### Concept

The bigram model is the simplest possible approach.  It maintains a **27×27
matrix of logits** where:

```
logits[i, j]  =  "how likely is character j to follow character i"
```

The forward pass is literally just an array index lookup:

```python
logits = self.logits[idx]   # shape (B, T, vocab_size)
```

No matrix multiplications, no activation functions, no layers — just a table.

### What it learns

After training, `logits[5, :]` (the row for the letter `e`) will have high
values at letters that commonly follow `e` in English names (like `r`, `n`,
`l`) and low values at rare ones.

### Limitation

It can only use **1 previous character** as context.  No matter how many
characters have been generated, the model only looks at the most recent one.
The bigram for "emm" is identical to the bigram for "am" — the `m` is all
that matters.


In [ ]:
class Bigram(nn.Module):
    '''
    A lookup table of logits: one row per input character, one column per
    possible next character.  The entire model is one (vocab_size, vocab_size)
    parameter matrix.
    '''

    def __init__(self, vocab_size):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(vocab_size, vocab_size))

    def get_block_size(self):
        return 1   # only needs 1 previous character

    def forward(self, idx, targets=None):
        # idx : (B, T)  — integer token indices
        logits = self.logits[idx]   # (B, T, vocab_size)  — row lookup

        loss = None
        if targets is not None:
            # Reshape to (B*T, vocab_size) and (B*T,) for cross_entropy
            # ignore_index=-1 masks the padding positions in Y
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1)
        return logits, loss


### Forward-pass walkthrough

Let's trace through a single batch so you can see every tensor shape.


In [ ]:
bigram = Bigram(vocab_size)

# grab a small batch
X_b, Y_b = train_dataset[:4]   # 4 examples

print('=== Bigram forward-pass walkthrough ===')
print(f'Input  idx shape : {X_b.shape}   (B=4, T={block_size})')
print()

with torch.no_grad():
    logits_b, loss_b = bigram(X_b, Y_b)

print(f'Output logits shape : {logits_b.shape}  (B=4, T={block_size}, vocab={vocab_size})')
print(f'Loss                : {loss_b.item():.4f}')
print()

# Show what the lookup actually does for the first example, first position
first_char_idx = X_b[0, 0].item()   # should be 0 (START token)
print(f'X_b[0,0] = {first_char_idx}  (the START token)')
print(f'logits[0,0] = bigram.logits[{first_char_idx}] =')
print(f'  shape {logits_b[0,0].shape}, first 5 values: {logits_b[0,0,:5].tolist()}')
print()
print('At init all logits are 0, so softmax gives uniform probs = 1/27 each.')
print(f'After training, logits[0] should peak at frequent first letters (a, e, m, ...).')


In [ ]:
print('Training Bigram for 2000 steps...')
bigram_model = Bigram(vocab_size)
steps_b, tr_b, te_b = train_model(
    bigram_model, train_dataset, test_dataset, DEVICE,
    max_steps=2000, batch_size=32, print_every=500)

print()
print('Samples from Bigram:')
print_samples(bigram_model, train_dataset, test_dataset, DEVICE, num=20)


In [ ]:
# Visualise the bigram table as a heatmap
with torch.no_grad():
    probs = F.softmax(bigram_model.logits, dim=-1).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(probs, cmap='Blues')
labels = ['<S>'] + list(chars)
ax.set_xticks(range(vocab_size)); ax.set_xticklabels(labels, fontsize=7)
ax.set_yticks(range(vocab_size)); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('next character'); ax.set_ylabel('current character')
ax.set_title('Bigram transition probabilities (after training)')
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.show()
print('Row i = given char i, what is the probability distribution over next chars.')


---
## Model 2 — MLP

*(Bengio et al., 2003)*

### Concept

The bigram sees only 1 previous character.  The MLP sees the last **`block_size`
characters** (a fixed context window).

Each character is mapped to a learned **embedding vector** of dimension
`n_embd`.  All embeddings in the context window are **concatenated** into one
long vector of size `block_size × n_embd`, which is then fed through a
single-hidden-layer neural network.

```
[char₁, char₂, char₃]
    ↓ embed (each → n_embd dims)
[e₁ | e₂ | e₃]         ← concatenated: (block_size × n_embd,)
    ↓ Linear → Tanh
hidden  (n_embd2,)
    ↓ Linear
logits  (vocab_size,)
```

### Why concatenate instead of sum?

Concatenation preserves **positional identity** — the network can learn
different weights for "what was in position 1" vs "what was in position 2".
Summing would collapse that information.

### Limitation

The context window is **fixed** at `block_size`.  If `block_size=3`, the model
can never use information from 4 characters back, no matter how relevant.


In [ ]:
class MLP(nn.Module):
    '''
    Fixed-context window model.  Embeds the last block_size characters,
    concatenates the embeddings, and predicts the next token with an MLP.
    '''

    def __init__(self, config):
        super().__init__()
        self.block_size = config['block_size']
        self.vocab_size = config['vocab_size']
        n_embd  = config['n_embd']
        n_embd2 = config['n_embd2']

        # +1 for a special <BLANK> token used when padding the left edge
        self.wte = nn.Embedding(self.vocab_size + 1, n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(self.block_size * n_embd, n_embd2),
            nn.Tanh(),
            nn.Linear(n_embd2, self.vocab_size),
        )

    def get_block_size(self):
        return self.block_size

    def forward(self, idx, targets=None):
        # idx : (B, T)
        embs = []
        for k in range(self.block_size):
            tok_emb = self.wte(idx)                     # (B, T, n_embd)
            idx     = torch.roll(idx, 1, 1)             # shift right by 1
            idx[:, 0] = self.vocab_size                 # left-pad with BLANK
            embs.append(tok_emb)

        # embs is a list of block_size tensors, each (B, T, n_embd)
        # concatenate along the last dimension
        x      = torch.cat(embs, dim=-1)               # (B, T, block_size*n_embd)
        logits = self.mlp(x)                            # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1), ignore_index=-1)
        return logits, loss


### Forward-pass walkthrough


In [ ]:
cfg = dict(vocab_size=vocab_size, block_size=block_size,
           n_embd=64, n_embd2=64)

mlp_demo = MLP(cfg)

X_b, Y_b = train_dataset[:4]
B, T = X_b.shape

print('=== MLP forward-pass walkthrough ===')
print(f'Input shape: {X_b.shape}  (B={B}, T={T})')
print()

with torch.no_grad():
    # Manually trace the forward pass
    idx_trace = X_b.clone()
    embs = []
    for k in range(block_size):
        tok_emb = mlp_demo.wte(idx_trace)
        print(f'  Step {k}: wte(idx) -> {tok_emb.shape}  '
              f'(B, T, n_embd={cfg["n_embd"]})')
        idx_trace = torch.roll(idx_trace, 1, 1)
        idx_trace[:, 0] = vocab_size  # BLANK
        embs.append(tok_emb)

    x = torch.cat(embs, dim=-1)
    print(f'  After cat(embs, dim=-1): {x.shape}  '
          f'(B, T, block_size*n_embd = {block_size}*{cfg["n_embd"]}={block_size*cfg["n_embd"]})')

    h = mlp_demo.mlp[0](x)      # Linear
    h = mlp_demo.mlp[1](h)      # Tanh
    print(f'  After Linear+Tanh     : {h.shape}  (B, T, n_embd2={cfg["n_embd2"]})')
    logits = mlp_demo.mlp[2](h) # Linear
    print(f'  After final Linear    : {logits.shape}  (B, T, vocab_size={vocab_size})')

print()
print('The MLP processes ALL T positions simultaneously in one batch.')
print('Each position gets a context of block_size previous characters.')


In [ ]:
print('Training MLP for 3000 steps...')
mlp_model = MLP(cfg)
steps_m, tr_m, te_m = train_model(
    mlp_model, train_dataset, test_dataset, DEVICE,
    max_steps=3000, batch_size=32, print_every=500)

print()
print('Samples from MLP:')
print_samples(mlp_model, train_dataset, test_dataset, DEVICE, num=20)


---
## Model 3 — Bag of Words (BoW)

### Concept

The MLP *concatenates* previous embeddings; the BoW *averages* them.

```
[e₁, e₂, e₃]  →  mean([e₁, e₂, e₃])  →  MLP  →  logits
```

This might seem like a step backwards (averaging loses order information), but
it reveals something profound: **averaging is just attention with uniform
weights**.

### The causal average

We can't let position 2 average position 3 (that would leak future
information).  So we use a **lower-triangular mask**:

```
position 0 averages: [e₀]
position 1 averages: [e₀, e₁]   / 2
position 2 averages: [e₀, e₁, e₂] / 3
…
```

In code:

```python
att = torch.zeros(B, T, T)
att = att.masked_fill(causal_mask == 0, float('-inf'))
att = F.softmax(att, dim=-1)   # uniform over allowed positions
y   = att @ x                  # weighted sum = average
```

This is **almost identical** to self-attention — the only difference is that
attention *learns* its weights while BoW uses fixed uniform weights.

### Why this matters

The BoW is a bridge: it proves that simple averaging already captures something
useful, and it shows exactly where Transformer attention comes from.

### Limitation

All positions get **equal weight**.  The model cannot learn "the first character
matters more than the third" — that requires learned weights, which is what the
Transformer's attention mechanism provides.


In [ ]:
class CausalBoW(nn.Module):
    '''
    Causal bag-of-words aggregation.
    Computes the average of all previous token embeddings at each position.
    The causal mask ensures position t cannot see positions > t.
    '''

    def __init__(self, block_size):
        super().__init__()
        # lower-triangular mask: entry (i,j)=1 means position i can see position j
        self.register_buffer(
            'bias',
            torch.tril(torch.ones(block_size, block_size)).view(1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        att = torch.zeros(B, T, T, device=x.device)
        att = att.masked_fill(self.bias[:, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)    # uniform over [0..t], zero elsewhere
        y   = att @ x                   # (B,T,T) @ (B,T,C) -> (B,T,C)
        return y


class BoWBlock(nn.Module):
    '''One BoW aggregation step followed by a two-layer MLP.'''

    def __init__(self, n_embd, n_embd2, block_size):
        super().__init__()
        self.cbow = CausalBoW(block_size)
        self.c_fc   = nn.Linear(n_embd,  n_embd2)
        self.c_proj = nn.Linear(n_embd2, n_embd)

    def forward(self, x):
        x = x + self.cbow(x)                              # residual
        x = x + self.c_proj(torch.tanh(self.c_fc(x)))    # residual MLP
        return x


class BoW(nn.Module):
    '''Full BoW language model.'''

    def __init__(self, config):
        super().__init__()
        vs  = config['vocab_size']
        bs  = config['block_size']
        d   = config['n_embd']
        d2  = config['n_embd2']
        self.block_size = bs
        self.vocab_size = vs
        self.wte   = nn.Embedding(vs, d)
        self.wpe   = nn.Embedding(bs, d)
        self.block = BoWBlock(d, d2, bs)
        self.lm_head = nn.Linear(d, vs)

    def get_block_size(self):
        return self.block_size

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos  = torch.arange(T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x    = self.wte(idx) + self.wpe(pos)    # (B, T, n_embd)
        x    = self.block(x)
        logits = self.lm_head(x)                # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1), ignore_index=-1)
        return logits, loss


### Walkthrough — comparing the causal average to attention


In [ ]:
bow_demo = BoW(cfg)
X_b, _ = train_dataset[:2]
B, T = X_b.shape

print('=== BoW forward-pass walkthrough ===')
print()

with torch.no_grad():
    pos = torch.arange(T, dtype=torch.long).unsqueeze(0)
    x   = bow_demo.wte(X_b) + bow_demo.wpe(pos)
    print(f'Token+position embeddings : {x.shape}  (B={B}, T={T}, n_embd={cfg["n_embd"]})')

    cbow = bow_demo.block.cbow
    att  = torch.zeros(B, T, T)
    att  = att.masked_fill(cbow.bias[:, :T, :T] == 0, float('-inf'))
    att  = F.softmax(att, dim=-1)
    print(f'Attention weight matrix   : {att.shape}')
    print()
    print('Attention weights for batch 0, positions 0-4:')
    print('(rows = query position, cols = key position)')
    print(att[0, :5, :5].numpy().round(3))
    print()
    print('Each row sums to 1.0 and is UNIFORM over allowed positions.')
    print('This is identical to self-attention — except the weights are FIXED,')
    print('not learned from Q and K projections.')
    print()

    y = att @ x
    print(f'After weighted sum (att @ x): {y.shape}')


In [ ]:
print('Training BoW for 3000 steps...')
bow_model = BoW(cfg)
steps_bow, tr_bow, te_bow = train_model(
    bow_model, train_dataset, test_dataset, DEVICE,
    max_steps=3000, batch_size=32, print_every=500)

print()
print('Samples from BoW:')
print_samples(bow_model, train_dataset, test_dataset, DEVICE, num=20)


---
## Model 4 — RNN (Recurrent Neural Network)

### Concept

Instead of processing a fixed window of characters all at once, the RNN
processes one character **at a time**, maintaining a **hidden state** `h` that
summarises everything seen so far.

```
h₀ = zeros                    ← initial hidden state
h₁ = tanh(W · [x₁, h₀])     ← after seeing char 1
h₂ = tanh(W · [x₂, h₁])     ← after seeing chars 1-2
h₃ = tanh(W · [x₃, h₂])     ← after seeing chars 1-3
…
logits_t = W_out · h_t
```

The hidden state `h` has dimension `n_embd2`.  It is updated by `RNNCell`,
which concatenates the current input embedding `x_t` with the previous hidden
state `h_{t-1}` and applies a linear layer + tanh:

```python
ht = tanh(W_xh · cat([xt, hprev]))
```

### Why tanh?

Tanh squashes outputs to `[-1, 1]`.  This prevents the hidden state from
growing unboundedly over many time steps.

### Limitation: gradient vanishing

Training an RNN with backpropagation-through-time requires gradients to flow
back through every `tanh` in the chain.  Since `|tanh'(x)| ≤ 1`, the gradient
shrinks with each step.  For long sequences the gradient effectively reaches
zero before it gets back to the early time steps — the model cannot learn
long-range dependencies.

The GRU (next model) addresses this directly.


In [ ]:
class RNNCell(nn.Module):
    '''
    Single RNN time step.
    Takes x_t (current input) and h_{t-1} (previous hidden state),
    returns h_t (new hidden state).
    '''

    def __init__(self, n_embd, n_embd2):
        super().__init__()
        # input:  cat([x_t, h_{t-1}])  shape (B, n_embd + n_embd2)
        # output: h_t                   shape (B, n_embd2)
        self.xh_to_h = nn.Linear(n_embd + n_embd2, n_embd2)

    def forward(self, xt, hprev):
        xh = torch.cat([xt, hprev], dim=1)   # (B, n_embd + n_embd2)
        ht = torch.tanh(self.xh_to_h(xh))    # (B, n_embd2)
        return ht


class RNN(nn.Module):

    def __init__(self, config, cell_type='rnn'):
        super().__init__()
        vs  = config['vocab_size']
        bs  = config['block_size']
        d   = config['n_embd']
        d2  = config['n_embd2']
        self.block_size = bs
        self.vocab_size = vs
        self.wte    = nn.Embedding(vs, d)
        self.start  = nn.Parameter(torch.zeros(1, d2))   # learnable initial h
        if cell_type == 'rnn':
            self.cell = RNNCell(d, d2)
        elif cell_type == 'gru':
            self.cell = GRUCell(d, d2)
        self.lm_head = nn.Linear(d2, vs)

    def get_block_size(self):
        return self.block_size

    def forward(self, idx, targets=None):
        B, T = idx.size()
        emb  = self.wte(idx)                      # (B, T, n_embd)
        hprev = self.start.expand(B, -1)          # (B, n_embd2)
        hiddens = []
        for i in range(T):
            xt = emb[:, i, :]                     # (B, n_embd)
            ht = self.cell(xt, hprev)             # (B, n_embd2)
            hprev = ht
            hiddens.append(ht)

        hidden = torch.stack(hiddens, dim=1)      # (B, T, n_embd2)
        logits = self.lm_head(hidden)             # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1), ignore_index=-1)
        return logits, loss


### Walkthrough — watching the hidden state evolve


In [ ]:
# Need GRUCell defined before RNN can use it with cell_type='gru'
# Define a placeholder so RNN instantiation works; real GRUCell comes next.
class GRUCell(nn.Module):
    def __init__(self, n_embd, n_embd2):
        super().__init__()
        self.xh_to_z    = nn.Linear(n_embd + n_embd2, n_embd2)
        self.xh_to_r    = nn.Linear(n_embd + n_embd2, n_embd2)
        self.xh_to_hbar = nn.Linear(n_embd + n_embd2, n_embd2)

    def forward(self, xt, hprev):
        xh = torch.cat([xt, hprev], dim=1)
        r  = torch.sigmoid(self.xh_to_r(xh))
        hprev_reset = r * hprev
        xhr  = torch.cat([xt, hprev_reset], dim=1)
        hbar = torch.tanh(self.xh_to_hbar(xhr))
        z    = torch.sigmoid(self.xh_to_z(xh))
        ht   = (1 - z) * hprev + z * hbar
        return ht

print('GRUCell defined (used by both RNN module with cell_type=gru and Model 5).')


In [ ]:
rnn_demo = RNN(cfg, cell_type='rnn')

word  = 'emma'
ix    = train_dataset.encode(word)   # tensor of ints
# wrap in batch of 1, prepend START token
x_single = torch.zeros(1, len(word)+1, dtype=torch.long)
x_single[0, 1:] = ix

print('=== RNN forward-pass walkthrough ===')
print(f'Word: {word!r}  ->  token ids: [0 (START), {ix.tolist()}]')
print()

with torch.no_grad():
    B, T = x_single.shape
    emb   = rnn_demo.wte(x_single)    # (1, T, n_embd)
    hprev = rnn_demo.start.expand(1, -1)

    for i in range(T):
        xt = emb[:, i, :]
        ht = rnn_demo.cell(xt, hprev)
        char_label = '<S>' if i == 0 else word[i-1]
        print(f'  t={i} ({char_label!r:3s})  xt:{tuple(xt.shape)}  ht:{tuple(ht.shape)}'
              f'  |h| mean={ht.abs().mean().item():.4f}')
        hprev = ht

print()
print('Key point: each h_t is the same shape regardless of sequence length.')
print('The hidden state compresses the ENTIRE history into n_embd2 numbers.')
print('Information from early timesteps must survive many tanh saturations.')


In [ ]:
print('Training RNN for 3000 steps...')
rnn_model = RNN(cfg, cell_type='rnn')
steps_r, tr_r, te_r = train_model(
    rnn_model, train_dataset, test_dataset, DEVICE,
    max_steps=3000, batch_size=32, print_every=500)

print()
print('Samples from RNN:')
print_samples(rnn_model, train_dataset, test_dataset, DEVICE, num=20)


---
## Model 5 — GRU (Gated Recurrent Unit)

*(Cho et al., 2014)*

### Why gates?

The vanilla RNN has one learnable recurrence:

```
h_t = tanh(W · [x_t, h_{t-1}])
```

Every time step overwrites the hidden state completely.  The GRU adds **two
gates** that give the model explicit control over what to keep and what to
update.

### The three computations

**1. Reset gate `r`** — how much of the previous hidden state to *forget*:

```
r = sigmoid(W_r · [x_t, h_{t-1}])    # (B, n_embd2)  values in (0,1)
```

**2. Candidate new state `h̄`** — what we *would* set h to if we reset:

```
h̄ = tanh(W_h · [x_t,  r ⊙ h_{t-1}])
```

(Using `r ⊙ h_{t-1}` means the reset gate can zero out parts of `h` before
computing the candidate.)

**3. Update gate `z`** — how much to *mix in* the candidate vs keep old:

```
z   = sigmoid(W_z · [x_t, h_{t-1}])
h_t = (1 - z) ⊙ h_{t-1}  +  z ⊙ h̄
```

When `z ≈ 0` → keep the old hidden state (the model says "nothing important
happened this step").
When `z ≈ 1` → replace with the candidate (the model says "update memory").

### Why this solves gradient vanishing

In the RNN the gradient must flow through every `tanh` gate.  In the GRU when
`z ≈ 0` the update `h_t = h_{t-1}` is nearly an **identity** — the gradient
flows straight back with no squashing.  The model can maintain gradient
highways through long sequences.


In [ ]:
# GRUCell was already defined above alongside the RNN walkthrough.
# Here we just confirm it and show the three gates explicitly.

gru_demo = RNN(cfg, cell_type='gru')   # GRU reuses the RNN wrapper

word = 'emma'
ix   = train_dataset.encode(word)
x_single = torch.zeros(1, len(word)+1, dtype=torch.long)
x_single[0, 1:] = ix

print('=== GRU forward-pass walkthrough (one word) ===')
print(f'Word: {word!r}')
print()

with torch.no_grad():
    emb   = gru_demo.wte(x_single)     # (1, T, n_embd)
    hprev = gru_demo.start.expand(1, -1)
    cell  = gru_demo.cell

    for i in range(x_single.size(1)):
        xt = emb[:, i, :]
        xh = torch.cat([xt, hprev], dim=1)

        r    = torch.sigmoid(cell.xh_to_r(xh))
        z    = torch.sigmoid(cell.xh_to_z(xh))
        xhr  = torch.cat([xt, r * hprev], dim=1)
        hbar = torch.tanh(cell.xh_to_hbar(xhr))
        ht   = (1 - z) * hprev + z * hbar

        char_label = '<S>' if i == 0 else word[i-1]
        print(f'  t={i} ({char_label!r:3s})'
              f'  r_mean={r.mean().item():.3f}'
              f'  z_mean={z.mean().item():.3f}'
              f'  |ht|={ht.abs().mean().item():.4f}')
        hprev = ht

print()
print('r close to 0 → reset (forget) most of previous hidden state')
print('r close to 1 → keep most of previous hidden state')
print('z close to 0 → h_t ≈ h_{t-1}  (identity shortcut, gradient flows!)')
print('z close to 1 → h_t ≈ h_bar    (full update)')


In [ ]:
print('Training GRU for 3000 steps...')
gru_model = RNN(cfg, cell_type='gru')
steps_g, tr_g, te_g = train_model(
    gru_model, train_dataset, test_dataset, DEVICE,
    max_steps=3000, batch_size=32, print_every=500)

print()
print('Samples from GRU:')
print_samples(gru_model, train_dataset, test_dataset, DEVICE, num=20)


---
## Model 6 — Transformer

*(Vaswani et al., 2017 — exactly as used in GPT-2)*

### The two problems the Transformer solves

**Problem 1 (vs RNN):** The RNN must process tokens one at a time.  Gradients
must travel through every step to reach the first token.  The Transformer
processes all tokens **in parallel** — every token attends to every previous
token directly in one operation, so gradients reach the first token in one hop.

**Problem 2 (vs BoW):** The BoW averages all previous embeddings uniformly.
The Transformer uses **learned attention weights**, so it can focus on the most
relevant previous tokens.

### Architecture

```
Input idx  (B, T)
   ↓ token embedding  wte   (vocab_size, n_embd)
   + position embedding wpe  (block_size, n_embd)
   ↓                               (B, T, n_embd)
 ┌─────────────────────────────┐
 │ Block × n_layer             │
 │  LayerNorm                  │
 │  CausalSelfAttention ──┐    │
 │  + residual           ┘    │
 │  LayerNorm                  │
 │  MLP (4× expand, GELU) ─┐  │
 │  + residual             ┘  │
 └─────────────────────────────┘
   ↓
 LayerNorm (ln_f)
   ↓
 Linear (lm_head): (B, T, vocab_size)
```

### Causal self-attention in detail

For each head (of `n_head` heads), with head dimension `hs = n_embd / n_head`:

```
Q = x W_Q    (B, T, hs)   ← what am I looking for?
K = x W_K    (B, T, hs)   ← what do I contain?
V = x W_V    (B, T, hs)   ← what do I output if attended to?

scores = Q K^T / sqrt(hs)  (B, T, T)
scores = mask_fill(scores, causal_mask==0, -inf)
weights = softmax(scores)   (B, T, T)
out = weights @ V           (B, T, hs)
```

The **causal mask** is lower-triangular: position `i` can only attend to
positions `0 … i`.  This prevents the model from "seeing the future" during
training.

**Multi-head**: run this attention `n_head` times in parallel with different
`W_Q, W_K, W_V` projections.  Concatenate results, project back to `n_embd`.

### Pre-norm residual stream

Every sub-layer (attention and MLP) is applied as a **residual**:

```
x = x + sublayer(LayerNorm(x))
```

The `LayerNorm` keeps activations stable.  The residual connection (`+ x`)
means gradients always have a direct path back to the input — another gradient
highway on top of the attention shortcut.


In [ ]:
class NewGELU(nn.Module):
    '''
    Gaussian Error Linear Unit — a smoother alternative to ReLU.
    Approximately x * Phi(x) where Phi is the Gaussian CDF.
    The formula here matches the original BERT/GPT implementation.
    '''
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))))


class CausalSelfAttention(nn.Module):
    '''
    Multi-head causal self-attention.

    Single combined linear layer produces Q, K, V for all heads at once
    (3 * n_embd outputs), then split and reshaped per head.
    '''

    def __init__(self, config):
        super().__init__()
        n_embd = config['n_embd']
        n_head = config['n_head']
        bs     = config['block_size']
        assert n_embd % n_head == 0, 'n_embd must be divisible by n_head'

        self.n_head = n_head
        self.n_embd = n_embd

        # Combined Q, K, V projection for all heads
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        # Output projection
        self.c_proj = nn.Linear(n_embd, n_embd)

        # Causal mask: (1, 1, block_size, block_size)
        self.register_buffer(
            'bias',
            torch.tril(torch.ones(bs, bs)).view(1, 1, bs, bs))

    def forward(self, x):
        B, T, C = x.size()   # C == n_embd
        nh, hs = self.n_head, C // self.n_head   # head size

        # Project to Q, K, V — all heads in one shot
        qkv = self.c_attn(x)                    # (B, T, 3*C)
        q, k, v = qkv.split(C, dim=2)           # each (B, T, C)

        # Reshape to (B, n_head, T, head_size)
        k = k.view(B, T, nh, hs).transpose(1, 2)
        q = q.view(B, T, nh, hs).transpose(1, 2)
        v = v.view(B, T, nh, hs).transpose(1, 2)

        # Attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))  # (B,nh,T,T)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)                               # (B,nh,T,T)

        # Weighted sum of values
        y = att @ v                              # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)   # (B, T, C)

        # Output projection
        y = self.c_proj(y)                       # (B, T, C)
        return y, att   # return att so we can visualise it


class Block(nn.Module):
    '''One Transformer block: pre-norm attention + pre-norm MLP, both residual.'''

    def __init__(self, config):
        super().__init__()
        n_embd  = config['n_embd']
        n_embd2 = config['n_embd2']   # used as the MLP hidden dim (usually 4*n_embd)
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.c_fc   = nn.Linear(n_embd,  4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.act    = NewGELU()

    def forward(self, x):
        attn_out, _ = self.attn(self.ln_1(x))
        x = x + attn_out                                          # residual
        x = x + self.c_proj(self.act(self.c_fc(self.ln_2(x))))   # residual MLP
        return x


class Transformer(nn.Module):
    '''GPT-2 style Transformer language model.'''

    def __init__(self, config):
        super().__init__()
        vs = config['vocab_size']
        bs = config['block_size']
        d  = config['n_embd']
        nl = config.get('n_layer', 4)

        self.block_size = bs
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(vs, d),
            wpe  = nn.Embedding(bs, d),
            h    = nn.ModuleList([Block(config) for _ in range(nl)]),
            ln_f = nn.LayerNorm(d),
        ))
        self.lm_head = nn.Linear(d, vs, bias=False)

        n_params = sum(p.numel() for p in self.transformer.parameters())
        print(f'Transformer parameters: {n_params/1e6:.2f}M')

    def get_block_size(self):
        return self.block_size

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos  = torch.arange(T, dtype=torch.long, device=idx.device).unsqueeze(0)

        tok_emb = self.transformer.wte(idx)   # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)   # (1, T, n_embd)
        x = tok_emb + pos_emb                 # (B, T, n_embd)

        for block in self.transformer.h:
            x = block(x)

        x      = self.transformer.ln_f(x)    # (B, T, n_embd)
        logits = self.lm_head(x)             # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1), ignore_index=-1)
        return logits, loss


### Forward-pass walkthrough — inside causal self-attention


In [ ]:
tf_cfg = dict(vocab_size=vocab_size, block_size=block_size,
              n_embd=64, n_embd2=64, n_head=4, n_layer=4)

tf_demo = Transformer(tf_cfg)
X_b, _ = train_dataset[:2]
B, T   = X_b.shape

print('=== Transformer forward-pass walkthrough ===')
print()

with torch.no_grad():
    pos     = torch.arange(T, dtype=torch.long).unsqueeze(0)
    tok_emb = tf_demo.transformer.wte(X_b)
    pos_emb = tf_demo.transformer.wpe(pos)
    x       = tok_emb + pos_emb

    print(f'Token embeddings  : {tok_emb.shape}  (B, T, n_embd)')
    print(f'Position embeddings: {pos_emb.shape}  (1, T, n_embd)')
    print(f'Sum (x)           : {x.shape}')
    print()

    # Manually run through the first block's attention
    block0 = tf_demo.transformer.h[0]
    x_ln   = block0.ln_1(x)
    print(f'After LayerNorm(x): {x_ln.shape}')

    n_embd = 64
    n_head = 4
    hs     = n_embd // n_head   # head size = 16

    qkv = block0.attn.c_attn(x_ln)           # (B, T, 3*64)
    q, k, v = qkv.split(n_embd, dim=2)       # each (B, T, 64)
    print(f'Q shape: {q.shape}   K shape: {k.shape}   V shape: {v.shape}')

    k = k.view(B, T, n_head, hs).transpose(1, 2)   # (B, n_head, T, hs)
    q = q.view(B, T, n_head, hs).transpose(1, 2)
    v = v.view(B, T, n_head, hs).transpose(1, 2)
    print(f'Q reshaped: {q.shape}  (B, n_head, T, head_size)')

    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
    print(f'Attention scores (before mask): {att.shape}  (B, n_head, T, T)')

    att = att.masked_fill(block0.attn.bias[:, :, :T, :T] == 0, float('-inf'))
    att = F.softmax(att, dim=-1)
    print(f'Attention weights (after softmax): {att.shape}')
    print()
    print(f'Attention weights for batch 0, head 0, positions 0-4:')
    print(att[0, 0, :5, :5].numpy().round(3))
    print()
    print('Key: position i has non-zero weight only for positions <= i (causal).')
    print('Unlike BoW, these weights are LEARNED - different for each head.')

    y = att @ v                                      # (B, n_head, T, hs)
    y = y.transpose(1, 2).contiguous().view(B, T, n_embd)
    print(f'After weighted sum + reshape: {y.shape}  (B, T, n_embd)')

    y = block0.attn.c_proj(y)
    print(f'After output projection: {y.shape}')


In [ ]:
print('Training Transformer for 3000 steps...')
tf_model = Transformer(tf_cfg)
steps_t, tr_t, te_t = train_model(
    tf_model, train_dataset, test_dataset, DEVICE,
    max_steps=3000, batch_size=32, print_every=500)

print()
print('Samples from Transformer:')
print_samples(tf_model, train_dataset, test_dataset, DEVICE, num=20)


In [ ]:
# Visualise attention weights from the trained transformer
tf_model.eval()
word_viz = 'isabella'
ix_viz   = train_dataset.encode(word_viz)
x_viz    = torch.zeros(1, len(word_viz)+2, dtype=torch.long, device=DEVICE)
x_viz[0, 1:1+len(ix_viz)] = ix_viz.to(DEVICE)
T_viz    = x_viz.size(1)

with torch.no_grad():
    pos  = torch.arange(T_viz, dtype=torch.long, device=DEVICE).unsqueeze(0)
    x_v  = tf_model.transformer.wte(x_viz) + tf_model.transformer.wpe(pos)
    # run through first block manually to grab attention weights
    b0   = tf_model.transformer.h[0]
    x_ln = b0.ln_1(x_v)
    _, att_weights = b0.attn(x_ln)   # (1, n_head, T, T)

labels_viz = ['<S>'] + list(word_viz) + ['<S>']
fig, axes  = plt.subplots(1, 4, figsize=(16, 4))
for h in range(4):
    ax = axes[h]
    data = att_weights[0, h, :T_viz, :T_viz].cpu().numpy()
    im = ax.imshow(data, vmin=0, vmax=1, cmap='Blues')
    ax.set_title(f'Head {h}')
    ax.set_xticks(range(T_viz)); ax.set_xticklabels(labels_viz[:T_viz], fontsize=8)
    ax.set_yticks(range(T_viz)); ax.set_yticklabels(labels_viz[:T_viz], fontsize=8)
    ax.set_xlabel('key (attended to)')
    ax.set_ylabel('query (attending from)')
plt.suptitle(f'Attention weights for "{word_viz}" — layer 0, all 4 heads', y=1.02)
plt.tight_layout()
plt.show()
print('Each head learns a different attention pattern.')
print('The lower-triangular structure shows the causal mask in action.')
tf_model.train()


---
## Final Comparison — All 6 Models

Let's put the test losses side by side and look at what each model generates.

*Note: all models were trained for only 3000 steps with default hyperparameters.
A proper training run uses 10 000–50 000 steps and hyperparameter tuning.
The ordering should hold, but the absolute numbers will be higher than optimal.*


In [ ]:
# Collect results (variables were set during training above)
models_info = [
    ('Bigram',      bigram_model,  steps_b,   tr_b,   te_b),
    ('MLP',         mlp_model,     steps_m,   tr_m,   te_m),
    ('BoW',         bow_model,     steps_bow, tr_bow, te_bow),
    ('RNN',         rnn_model,     steps_r,   tr_r,   te_r),
    ('GRU',         gru_model,     steps_g,   tr_g,   te_g),
    ('Transformer', tf_model,      steps_t,   tr_t,   te_t),
]

print(f'{'Model':<14}  Final train loss  Final test loss')
print('-' * 48)
for name, _, _, tr, te in models_info:
    print(f'{name:<14}  {tr[-1]:.4f}            {te[-1]:.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, _, steps, _, te in models_info:
    ax.plot(steps, te, marker='o', label=name)

ax.set_xlabel('Training step')
ax.set_ylabel('Test loss (cross-entropy)')
ax.set_title('Test loss over training — all 6 models')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print('=' * 60)
print('SAMPLES FROM ALL 6 MODELS')
print('=' * 60)
for name, model, *_ in models_info:
    print(f'\n--- {name} ---')
    print_samples(model, train_dataset, test_dataset, DEVICE, num=10)


## Summary — what each model adds

| Model | Key addition | What it fixes |
|-------|-------------|---------------|
| Bigram | Learnable 27×27 transition table | — (baseline) |
| MLP | Embeddings + fixed context window + hidden layer | Richer representations, N-char context |
| BoW | Causal average over all previous embeddings | Unlimited context (uniform) |
| RNN | Sequential hidden state | Unlimited context (compressed) |
| GRU | Reset + update gates | Gradient vanishing in long sequences |
| Transformer | Learned multi-head self-attention | All context, learned weights, parallel |

The Transformer is not magic — it is a systematic solution to the weaknesses of
all the models before it:
* Unlike Bigram/MLP: **unlimited context**.
* Unlike BoW: **learned weights** (not uniform), so it can focus on what matters.
* Unlike RNN/GRU: **parallel processing**, so gradients reach every token in
  one hop regardless of sequence length.

The same architecture, scaled up by 4-5 orders of magnitude, is GPT-4.
